### Prueba Pipeline Google Cloud


#### Librerías y Path Variables

In [ ]:
# Configuración de recursos según la máquina
entorno = 'nube' # cambiar a 'casa' u 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09

elif entorno == 'nube':
    import os, psutil, socket
    cpus = os.cpu_count()
    ram_gb = int(psutil.virtual_memory().total / 1024**3)
    vm_nombre = socket.gethostname()
    
    duckdb_threads = min(cpus, 8)
    duckdb_memory_limit = f'{int(ram_gb * 0.6)}GB'
    rf_n_jobs = min(cpus, 8)
    lgb_threads = min(cpus, 8)
    
    print(f"VM: {vm_nombre} | {cpus} vCPU | {ram_gb} GB RAM")
    
else:
    raise ValueError("entorno debe ser 'casa', 'oficina' o 'nube'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

In [ ]:
# Importando Librerías

import os, sys
# os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import StratifiedShuffleSplit

import lightgbm as lgb

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour


In [ ]:
# Paths Variables

from pathlib import Path
cwd = Path.cwd()

sys.path.insert(0, str(cwd))                      # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

RAIZ = Path('/home/ds/buckets/b1/01_Prueba')      # CAMBIO: datos en el bucket
base_path = str(RAIZ) + '/'
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
modelos_path = base_path + 'modelos/'
db_path = base_path + 'data/databases/'
dataset_file = 'prueba_4meses.parquet'            # CAMBIO: parquet generado desde analistasr

#### Loading Data and Input Variables

##### Cargado de Datos

In [ ]:
# Función para cargar los datos, filtrando por meses

ruta_parquet = dataset_path + dataset_file    # .../data/prueba_4meses.parquet
ROTA = 'ccajas_depositos'
ROTA_CONFIRMADA = True                        # confirmado también en este dataset: 202105 en cero
con = duckdb.connect()

def cargar(meses):                            # CAMBIO: sin archivo_cols, se leen todas las columnas
    lista_meses = ', '.join(str(m) for m in meses)

    df = con.execute(f"""
        select * from read_parquet('{ruta_parquet}')
        where foto_mes in ({lista_meses})
    """).fetchdf()

  # la variable rota: NaN solo en mayo
    if ROTA_CONFIRMADA and ROTA in df.columns:
        df.loc[df['foto_mes'] == 202105, ROTA] = np.nan

  # columnas que llegan como texto desde el CSV, pero son 0/1                     # CAMBIO
    for c in ['tmobile_app', 'cmobile_app_trx']:                                  # CAMBIO
        df[c] = df[c].astype(int)                                                 # CAMBIO

    return df

In [ ]:
# Cargar los datos, filtrando por meses

MESES = [202103, 202105, 202106, 202108]

data = cargar(MESES)                                           # CAMBIO: todas las columnas, sin .txt

data = data.sort_values(['foto_mes', 'numero_de_cliente'])     # orden fijo de filas

print(data.shape)
print(data['foto_mes'].value_counts().sort_index())

##### Ganancia/Costo y Abril/Junio para Train/Test 

In [ ]:
# Ganancia y Costo y Selección de Meses para Entrenamiento y Test

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202103    # Marzo - mes de entrenamiento
mes_test = 202105     # Mayo - mes de testeo

##### Clase Binaria + X/Y_Train/Test

In [ ]:
data['clase_binaria_b2'] = np.where(data['clase_ternaria'] == 'BAJA+2', 1, 0)    # optimizo y mido con esta


# Preparación de datos para la optimización
cols_descartar = ['clase_ternaria', 'clase_binaria_b2', 'numero_de_cliente', 'foto_mes']

train_data = data[data['foto_mes'] == mes_train]
test_data  = data[data['foto_mes'] == mes_test]

X_train = train_data.drop(columns=cols_descartar)
y_train_binaria = train_data['clase_binaria_b2']

X_test = test_data.drop(columns=cols_descartar)
y_test_binaria = test_data['clase_binaria_b2']        # ← era 'clase_binaria'
y_test_class = test_data['clase_ternaria']            # solo para medir; no entra en X

train_data_lgb = lgb.Dataset(X_train, label=y_train_binaria)
test_data_lgb  = lgb.Dataset(X_test, label=y_test_binaria, reference=train_data_lgb)

#### Optimización de Hiperparámetros (con datos abril y cross-validation)

In [ ]:
sss_opt = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=semillas[0])   # stratified shuffle split para la optimización bayesiana 
                                                                                        # con 5 splits y 30% de testeo

def objective(trial):

    # bayesianos
    num_leaves = trial.suggest_int('num_leaves', 8, 256)                                # numero de hojas del árbol
    min_sum_hessian_in_leaf = trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 100, log=True)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.2, log=True)          # mas bajo, más iteraciones necesita
    feature_fraction = trial.suggest_float('feature_fraction', 0.3, 0.7)                # columnas por árbol
    feature_fraction_bynode = trial.suggest_float('feature_fraction_bynode', 0.1, 0.5)  # columnas por nodo (se multiplica con la anterior)
    bagging_fraction = trial.suggest_float('bagging_fraction', 0.6, 1.0)                # submuestreo de filas por árbol
    bagging_freq = trial.suggest_int('bagging_freq', 1, 10)                             # > 0 para que el bagging funcione

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'boost_from_average': True,
        'feature_pre_filter': False,        
      # fijos de la cátedra
        'max_bin': 31,                       # número de bins para discretizar las variables continuas
        'min_data_in_leaf': 0,               # el freno pasa a min_sum_hessian_in_leaf
      # bayesianos
        'num_leaves': num_leaves,
        'min_sum_hessian_in_leaf': min_sum_hessian_in_leaf,
        'learning_rate': learning_rate,
        'feature_fraction': feature_fraction,
        'feature_fraction_bynode': feature_fraction_bynode,
        'bagging_fraction': bagging_fraction,
        'bagging_freq': bagging_freq,
        'seed': semillas[0],
        'num_threads': lgb_threads,
        'verbose': -1
    }
    num_boost_round = 3000                   # subido de 3000 porque el learning_rate mínimo es más bajo
    cv_results = lgb.cv(
        params,
        train_data_lgb,
        num_boost_round=num_boost_round,
        callbacks=[lgb.early_stopping(stopping_rounds=min(int(50 + 5 / learning_rate), num_boost_round // 3), verbose=False)],  
        folds=sss_opt.split(X_train, y_train_binaria),                    # usamos StratifiedShuffleSplit en vez del k-fold interno
        seed=semillas[0]
    )

    max_auc = max(cv_results['valid auc-mean'])
    best_iter = cv_results['valid auc-mean'].index(max_auc) + 1     # es el num_iterations

    # Guardamos cual es la mejor iteración del modelo
    trial.set_user_attr("best_iter", best_iter)

    return max_auc

storage_name = "sqlite:///" + db_path + "cpt2_optim_lgbm_Step1.db"
study_name = "cpt2_step1_b2_lgbm_sss_3000_con_FE04-b"                            # se hizo con FE04b es decir con deltas además de intrames

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

print(len(study.trials))     # debe dar 0 si la base es nueva

In [ ]:
objetivo_total = 50                                        # número de trials que queremos correr en total
faltan = objetivo_total - len(study.trials)
print(f"Tengo {len(study.trials)}, corro {faltan} más")
if faltan > 0:
    study.optimize(objective, n_trials=faltan)

#### Ganancia Acumulada

##### BAJA+2 + BAJA+1

**Target Fusionado (BAJA+1 y BAJA+2): Entrenamiento en Marzo y Prueba en Mayo con el Trial 25**

In [ ]:
# Target fusionado: BAJA+1 y BAJA+2 juntos (1) contra CONTINUA (0)
data['clase_binaria_b12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)

# Marzo para entrenar y mayo para testear                                         # CAMBIO: comentario
train_data2 = data[data['foto_mes'] == mes_train]
test_data2 = data[data['foto_mes'] == mes_test]

# Se descartan el target, el auxiliar del BAJA+2 solo, el id y el mes, para que no entren a X
cols_a_descartar = ['clase_ternaria', 'clase_binaria_b2', 'clase_binaria_b12', 'numero_de_cliente', 'foto_mes']

X_train2 = train_data2.drop(columns=cols_a_descartar)
y_train_binaria2 = train_data2['clase_binaria_b12']

X_test2 = test_data2.drop(columns=cols_a_descartar)
y_test_binaria2 = test_data2['clase_binaria_b12']
y_test_class2 = test_data2['clase_ternaria']          # solo para medir con el BAJA+2 real; no entra en X

train_data_lgb2 = lgb.Dataset(X_train2, label=y_train_binaria2)

# Receta del trial 25: sus hiperparámetros y el número de árboles (best_iter) que guardó la optimización
trial_25 = next(t for t in study.trials if t.number == 25)   
best_iter = trial_25.user_attrs['best_iter']
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'first_metric_only': True,
    'boost_from_average': True,
    'feature_pre_filter': False,
    'max_bin': 31,                                         # fijo de la cátedra
    'min_data_in_leaf': 0,                                 # fijo de la cátedra: el freno es min_sum_hessian_in_leaf
    **trial_25.params,                                     # los 7 hiperparámetros de la optimización
    'seed': semillas[0],
    'num_threads': lgb_threads,
    'verbose': -1
}

# Entrena con la primera semilla, para esta prueba rápida (las 20 semillas se hacen en la celda de abajo)
model_baja1_2 = lgb.train(
    {**params, 'seed': semillas[0]},
    train_data_lgb2,
    num_boost_round=best_iter
)

# Probabilidades de mayo del modelo fusionado                                     # CAMBIO: comentario
y_pred_baja1_2 = model_baja1_2.predict(X_test2)

# Ganancia con el BAJA+2 real: se acierta solo si el cliente fue BAJA+2, aunque el modelo haya aprendido las dos bajas
def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

ganancia_nueva = ganancia_real(y_pred_baja1_2, y_test_class2.values)
print(f"Ganancia en Mayo (target BAJA+1+BAJA+2 vs CONTINUA): {ganancia_nueva:.2f}")   # CAMBIO: junio -> mayo

**Curva de Ganancia Acumulada en Mayo (Target BAJA+1+BAJA+2, Trial 25 (OptimValid AbrilJunio), una semilla)**

In [ ]:
import plotly.graph_objects as go

# Ordena los clientes de mayo de mayor a menor probabilidad del modelo fusionado
orden2 = np.argsort(y_pred_baja1_2)[::-1]
# Resultado real de cada cliente en ese orden. Se usa y_test_binaria (solo BAJA+2), no y_test_binaria2, para medir con la ganancia real
ganancia_ordenada2 = np.where(y_test_binaria.values == 1, ganancia_acierto, -costo_estimulo)[orden2]
# Ganancia acumulada: lo que se gana si se estimulan los primeros 1, 2, 3... clientes
ganancia_acumulada2 = np.cumsum(ganancia_ordenada2)

# Mejor corte: el N donde la ganancia acumulada llega a su máximo (+1 porque argmax empieza en 0)
mejor_corte2 = np.argmax(ganancia_acumulada2) + 1
mejor_ganancia2 = ganancia_acumulada2[mejor_corte2 - 1]
# Probabilidad del último cliente que entra en ese corte (en el fusionado la escala es más alta que en el BAJA+2 solo)
p_corte2 = y_pred_baja1_2[orden2][mejor_corte2 - 1]

limite = 20000  # cuantos clientes mostrar en el eje x

fig = go.Figure()

# Curva de ganancia acumulada, hasta el límite del eje
fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada2[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

# Línea vertical punteada en el corte óptimo
fig.add_vline(
    x=mejor_corte2, line_dash='dash', line_color='black',
    annotation_text=f'Corte óptimo: {mejor_corte2} clientes',
    annotation_position='top'
)

# Punto rojo en el máximo, con la probabilidad de corte al lado
fig.add_trace(go.Scatter(
    x=[mejor_corte2], y=[mejor_ganancia2],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte2:.3f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title='Ganancia acumulada según cantidad de clientes estimulados (target BAJA+1+BAJA+2) - Trial 25',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia2:.2f} en el corte {mejor_corte2}, con probabilidad de corte p = {p_corte2:.4f}")

**Fusionado (BAJA+1+BAJA+2): 20 Semillas, Mejor Corte y Ganancia por Corte en Junio (Trial 25)**

In [ ]:
##### Fusionado (BAJA+1+BAJA+2): 20 semillas, mejor corte y ganancia por corte en Mayo (Trial 25)
#### NO SE CORRE SI SE LEVANTA DEL PICKLE GENERADO POR ESTA MISMA CELDA ANTES

n_semillas = 20                                   # cantidad de semillas
cortes = np.arange(4000, 20001, 500)              # cortes fijos (4.000, 4.500, ... 20.000) para el boxplot por corte
carpeta_modelos = modelos_path + 'cpt05_modelos_TrainMar_FE04b_sinopt/'   # CAMBIO: carpeta nueva, no pisa la de abril
os.makedirs(carpeta_modelos, exist_ok=True)                               # CAMBIO: la crea, porque se salteó la celda que la creaba

rows = []                                         # ganancia de cada semilla en cada corte fijo
rows_N = []                                       # mejor N y su ganancia, una fila por semilla
resultados_junio_b12 = {}                         # probabilidades de MAYO de cada modelo fusionado: (trial, semilla) -> probabilidades   # CAMBIO: comentario (el nombre de la variable no cambia)

for seed in semillas[:n_semillas]:
    params_seed = {**params, 'seed': seed}        # configuración del trial 25, con la semilla de esta vuelta

    # Entrena con marzo (target fusionado), guarda el modelo y predice mayo                    # CAMBIO: comentario
    model_seed = lgb.train(params_seed, train_data_lgb2, num_boost_round=best_iter)
    model_seed.save_model(carpeta_modelos + f"cpt05_TrainMar-TestMay-FE04b_modb12_t25_s{seed}.txt")    # CAMBIO: nombre
    y_pred_seed = model_seed.predict(X_test2)
    resultados_junio_b12[(25, seed)] = y_pred_seed

    # Ordena los clientes de mayo de mayor a menor probabilidad                                 # CAMBIO: comentario
    orden_seed = np.argsort(y_pred_seed)[::-1]
    # Resultado real de cada cliente en ese orden, medido con el BAJA+2 real (y_test_binaria): +ganancia si fue BAJA+2, -costo si no
    ganancia_ordenada_seed = np.where(y_test_binaria.values[orden_seed] == 1, ganancia_acierto, -costo_estimulo)
    # Ganancia acumulada: lo que se gana si se estimulan los primeros 1, 2, 3... clientes
    ganancia_acumulada_seed = np.cumsum(ganancia_ordenada_seed)

    # Mejor corte de esta semilla: el N donde la ganancia acumulada llega a su máximo (+1 porque argmax empieza en 0)
    mejor_N = int(np.argmax(ganancia_acumulada_seed)) + 1
    rows_N.append({"seed": seed, "mejor_N": mejor_N, "mejor_ganancia": ganancia_acumulada_seed[mejor_N - 1]})

    # Ganancia de esta semilla en cada corte fijo (-1 porque la posición N está en el índice N-1)
    for corte in cortes:
        rows.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada_seed[corte - 1]
        })

# Guarda las probabilidades de los 20 modelos (sirven con el orden de filas fijo que ya tiene data)
with open(modelos_path + 'cpt05_TrainMar-TestMay-FE04b_probb12_t25.pkl', 'wb') as f:      # CAMBIO: nombre
    pickle.dump(resultados_junio_b12, f)

# Mejor N por semilla: de acá sale el N central (la mediana) que se pasa a agosto
df_mejorN_b12 = pd.DataFrame(rows_N)
print("Mediana del mejor N (fusionado):", df_mejorN_b12['mejor_N'].median())
print("Mediana de la ganancia máxima:", df_mejorN_b12['mejor_ganancia'].median())

# Ganancia por semilla en cada corte fijo: alimenta las estadísticas y el boxplot
df_ganancia_cortes = pd.DataFrame(rows)

# Estadísticas por corte sobre las 20 semillas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes

In [ ]:
##### Recupera los resultados de las 20 semillas de Mayo desde el .pkl (sin reentrenar)

with open(modelos_path + 'cpt05_TrainMar-TestMay-FE04b_probb12_t25.pkl', 'rb') as f:
    resultados_junio_b12 = pickle.load(f)         # probabilidades de mayo por (trial, semilla). Mismo orden de filas, porque data se ordena al cargar

cortes = np.arange(4000, 20001, 500)              # mismos cortes fijos de antes

rows = []                                         # ganancia de cada semilla en cada corte fijo
rows_N = []                                       # mejor N y su ganancia, una fila por semilla

for (trial, seed), y_pred_seed in resultados_junio_b12.items():
    orden_seed = np.argsort(y_pred_seed)[::-1]
    ganancia_ordenada_seed = np.where(y_test_binaria.values[orden_seed] == 1, ganancia_acierto, -costo_estimulo)
    ganancia_acumulada_seed = np.cumsum(ganancia_ordenada_seed)

    mejor_N = int(np.argmax(ganancia_acumulada_seed)) + 1
    rows_N.append({"seed": seed, "mejor_N": mejor_N, "mejor_ganancia": ganancia_acumulada_seed[mejor_N - 1]})

    for corte in cortes:
        rows.append({"seed": seed, "corte": corte, "ganancia": ganancia_acumulada_seed[corte - 1]})

df_mejorN_b12 = pd.DataFrame(rows_N)
df_ganancia_cortes = pd.DataFrame(rows)

n_semillas = 20                                   # CAMBIO: cantidad de semillas (se definía en la celda del entrenamiento, que ya no se corre)

# Control: tiene que dar lo mismo que la corrida original
print("Mediana del mejor N (fusionado):", df_mejorN_b12['mejor_N'].median())              # esperado: 8079.5
print("Mediana de la ganancia máxima:", df_mejorN_b12['mejor_ganancia'].median())        # esperado: 273322500.0

**Boxplot de Ganancia por Corte en Mayo (Fusionado, 20 semillas, Trial 25 Optim/Valid Abril/Junio)**

In [ ]:
# Estadísticas por corte sobre las 20 semillas, más q1 y q3 (percentiles 25 y 75) para las cajas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

# Un boxplot por corte, armado con las estadísticas ya calculadas
fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],         # corte como texto: cada uno es una caja separada
    q1=estadisticas_cortes['q1'],                          # borde inferior de la caja
    median=estadisticas_cortes['median'],                  # línea del medio
    q3=estadisticas_cortes['q3'],                          # borde superior de la caja
    lowerfence=estadisticas_cortes['min'],                 # bigote inferior: mínimo de las 20 semillas
    upperfence=estadisticas_cortes['max'],                 # bigote superior: máximo de las 20 semillas
    mean=estadisticas_cortes['mean'],                      # media (línea punteada)
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte - Fusionado BAJA+1+BAJA+2 ({n_semillas} semillas) - Trial 25',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()

**N de Agosto a partir del Mejor Corte de Mayo (Fusionado BAJA+1+BAJA+2, Trial 25)**

In [ ]:
##### N de Agosto a partir del mejor corte de Mayo

# Cantidad de clientes de cada mes: de mayo (test) y de agosto (el mes a predecir)
clientes_junio = len(y_test_binaria)                          # CAMBIO: comentario. Ahora es mayo: 163768
clientes_agosto = int((data['foto_mes'] == 202108).sum())     # 164647

# N central en mayo: mediana de los 20 mejores N (uno por semilla) del fusionado, y su peso como % de los clientes de mayo
N_central_junio = df_mejorN_b12['mejor_N'].median()
pct_central = N_central_junio / clientes_junio

# Barrido: el mismo porcentaje central, corrido -20%, -10%, 0, +10% y +20%
barrido = pd.DataFrame({'variacion': [-0.20, -0.10, 0.0, 0.10, 0.20]})
barrido['pct'] = (pct_central * (1 + barrido['variacion'])).round(4)

# Cada porcentaje se traslada a agosto: N de clientes a estimular en cada submit
barrido['N_agosto'] = (barrido['pct'] * clientes_agosto).round().astype(int)

print(f"Mediana del mejor N en mayo (fusionado): {N_central_junio:.0f}  ->  {pct_central*100:.2f}% de {clientes_junio} clientes")    # CAMBIO: junio -> mayo
print(f"Clientes en agosto: {clientes_agosto}")
barrido

#### Entrenamiento Final: Junio (Target Fusionado) para Predecir Agosto

In [ ]:
# Entrenamiento final: junio (con etiquetas) para predecir agosto (sin etiquetas)
mes_train_jun = 202106
mes_pred_ago = 202108

train_jun = data[data['foto_mes'] == mes_train_jun]
pred_ago = data[data['foto_mes'] == mes_pred_ago]

# IDs de los clientes a predecir, guardados ANTES del drop: son los que van al archivo del submit, en el mismo orden que las predicciones
ids_pred_ago = pred_ago['numero_de_cliente'].values

# Mismas columnas descartadas que en abril-junio; el target es el fusionado (cols_a_descartar viene de la celda anterior)
X_train_jun = train_jun.drop(columns=cols_a_descartar)
y_train_jun = train_jun['clase_binaria_b12']
X_pred_ago = pred_ago.drop(columns=cols_a_descartar)

train_data_lgb_jun = lgb.Dataset(X_train_jun, label=y_train_jun)

# Chequeos: mismas columnas que en el entrenamiento de marzo (X_train2), 164.647 clientes a predecir, y cuántos positivos hay en junio    # CAMBIO: abril -> marzo
print(X_train_jun.shape, X_pred_ago.shape, len(ids_pred_ago))
print((X_train_jun.columns == X_train2.columns).all())
print(y_train_jun.value_counts())

**Agosto: 20 Semillas Nuevas con la Receta del Trial 25 (Target Fusionado), Entrenadas con Junio**

In [ ]:
##### Agosto: 20 semillas nuevas con la receta del trial 25 (target fusionado), entrenadas con junio

# 20 semillas nuevas, distintas de las usadas en la validación (semillas[:20]); hacen falta al menos 40 semillas
assert len(semillas) >= 40
semillas_pred = semillas[20:40]

carpeta_modelos_pred = modelos_path + 'cpt05_modelos_TrainJun_FE04b_sinopt/'      # CAMBIO: carpeta nueva, no pisa los modelos del envío final (cpt04)
os.makedirs(carpeta_modelos_pred, exist_ok=True)

resultados_pred_b12 = {}                                  # probabilidades de agosto de cada modelo: (trial, semilla) -> probabilidades

for seed in semillas_pred:
    params_seed = {**params, 'seed': seed}                # receta del trial 25 (params y best_iter de la celda del fusionado)

    # Entrena con junio (target fusionado), guarda el modelo y predice agosto
    model_seed = lgb.train(params_seed, train_data_lgb_jun, num_boost_round=best_iter)
    model_seed.save_model(carpeta_modelos_pred + f"cpt05_TrainJun-PredAgo-FE04b_modb12_t25_s{seed}.txt")   
    resultados_pred_b12[(25, seed)] = model_seed.predict(X_pred_ago)
    print(f'semilla {seed} lista')

# Guarda las probabilidades y los IDs, en el mismo orden (van juntos para armar los submits)
with open(modelos_path + 'cpt05_TrainJun-_PredAgo-FE04b_probb12_t25.pkl', 'wb') as f:     
    pickle.dump(resultados_pred_b12, f)
np.save(modelos_path + 'cpt05_TrainJun-FE04b_PredAgo_ids_pred.npy', ids_pred_ago)         

print(len(resultados_pred_b12), 'modelos entrenados con junio para predecir agosto')

**Archivos para el bot: 20 semillas por N y semillerío (trial 25, FE04b optimizado)**

In [ ]:
# Archivos para el bot: una lista de IDs por cada (N, semilla), más el semillerío de cada N

nro_submit = '05'            # número de submit
meses_sub = 'mmja'            # meses (ej. aja = abril-junio-agosto) y otras variantes
fe_sub = 'FE05bsinOpt'          # dataset + otra información (ej. FE04bOpt = con deltas + optimización)

hacer_20 = False             # True = genera los 20 archivos (uno por semilla) por cada N
hacer_sem = True             # True = genera el semillerío (un archivo) por cada N

variaciones = [-0.2, -0.1, 0.0, 0.1, 0.2]   # cortes a generar. Todos: [-0.2, -0.1, 0.0, 0.1, 0.2]. Solo el central: [0.0]
N_central_manual = None                 # N central de agosto a mano (ej. 14,000). None = usa el del barrido
Ns_manual = None                      # lista de N de agosto a mano. None = usa variaciones / N_central_manual

pref_carpeta = f'submit{nro_submit}{meses_sub}{fe_sub}'      # da 'submit05mmjaFE04bOpt'
pref_archivo = f'Sbm{nro_submit}{meses_sub}{fe_sub}'         # da 'Sbm05mmjaFE04bOpt'

carpeta_subs = dataset_path + 'out/'                    # los submits van en data/out

if Ns_manual is not None:               # CAMBIO: si hay lista a mano, manda esta y se ignora lo demás
    Ns = [int(n) for n in Ns_manual]
elif N_central_manual is None:          # usa el barrido calculado en la celda del N de agosto
    Ns = barrido.loc[barrido['variacion'].round(2).isin(variaciones), 'N_agosto'].tolist()
else:                                   # central a mano; los demás cortes salen de él
    Ns = [int(round(N_central_manual * (1 + v))) for v in variaciones]
print('N a generar:', Ns)               # para verlos antes de escribir

def escribir_ids(ruta, ids):
  # IDs como enteros (nunca notación científica), uno por línea, sin encabezado
    with open(ruta, 'w') as f:
        f.write('\n'.join(str(int(c)) for c in ids))

# Probabilidades de agosto de las 20 semillas, cliente por cliente, en el mismo orden que ids_pred_ago
p_semillas = {seed: resultados_pred_b12[(25, seed)] for seed in semillas_pred}      # 34 -> 25
assert all(len(p) == len(ids_pred_ago) for p in p_semillas.values())

# Semillerío: promedio de las 20 probabilidades, cliente por cliente
p_prom = np.mean(list(p_semillas.values()), axis=0)

for N in Ns:
  # Opción A: 20 archivos, uno por semilla, todos con el mismo N
    if hacer_20:
        carpeta_N = carpeta_subs + f'{pref_carpeta}N{N}/'
        os.makedirs(carpeta_N, exist_ok=True)
        for i, (seed, p) in enumerate(p_semillas.items(), start=1):   # i = 1..20
            idx = np.argsort(p)[::-1][:N]               # los N clientes con mayor probabilidad
            escribir_ids(carpeta_N + f'{pref_archivo}N{N}_s{i:02d}.csv', ids_pred_ago[idx])

  # Opción B: semillerío, un solo archivo por N (carpeta aparte para no mezclarlo con los 20)
    if hacer_sem:
        carpeta_sem = carpeta_subs + f'{pref_carpeta}N{N}Sem/'
        os.makedirs(carpeta_sem, exist_ok=True)
        idx = np.argsort(p_prom)[::-1][:N]
        escribir_ids(carpeta_sem + f'{pref_archivo}N{N}_sem.csv', ids_pred_ago[idx])

# Chequeos antes de mandar: solo de lo que se generó
for N in Ns:
    if hacer_20:
        carpeta_N = carpeta_subs + f'{pref_carpeta}N{N}/'
        archivos = sorted(os.listdir(carpeta_N))
        ok_lineas, ok_e, ok_unicos = True, True, True
        for a in archivos:
            texto = open(carpeta_N + a).read()
            ids = texto.split('\n')
            ok_lineas = ok_lineas and len(ids) == N         # tiene que tener exactamente N líneas
            ok_e = ok_e and ('e+' not in texto)             # sin notación científica
            ok_unicos = ok_unicos and len(set(ids)) == len(ids)   # sin IDs repetidos
        print(N, '| 20 archivos:', len(archivos) == 20, '| N líneas:', ok_lineas, '| sin e+:', ok_e, '| IDs únicos:', ok_unicos)

    if hacer_sem:
        texto = open(carpeta_subs + f'{pref_carpeta}N{N}Sem/' + f'{pref_archivo}N{N}_sem.csv').read()
        ids = texto.split('\n')
        print(N, '| semillerío | N líneas:', len(ids) == N, '| sin e+:', 'e+' not in texto, '| IDs únicos:', len(set(ids)) == len(ids))

In [ ]:
##### Comparación con el envío final (sin enviar nada)

p_prom = np.mean([resultados_pred_b12[(25, s)] for s in semillas_pred], axis=0)     # semillerío de las 20 semillas nuevas
orden = np.argsort(p_prom)[::-1]                                                     # clientes de mayor a menor probabilidad
ids_nuevos = ids_pred_ago[orden]

archivo_enviado = dataset_path + 'out/submit04ajaFE04bOptN12000Sem_final/Sbm04ajaFE04bOptN12000_sem_final.csv'          # CSV del envío final: 12.000 IDs, en orden de probabilidad
print("CSV final existe:", os.path.exists(archivo_enviado))
ids_enviados = np.array(Path(archivo_enviado).read_text().split('\n'), dtype=np.int64)

for N in [8117, 12000]:
    en_comun = len(set(ids_nuevos[:N]) & set(ids_enviados[:N]))
    print(N, '| en común:', en_comun, 'de', N, '| idénticos en el mismo orden:', np.array_equal(ids_nuevos[:N], ids_enviados[:N]))